# WebNLG verbalisation evaluation

This notebook reads generation CSVs and evaluates each generated verbalisation against **all references** attached to the entry.

Metrics computed per instance:
- **ROUGE-L F1**
- **METEOR**
- **chrF++**
- **BERTScore F1**
- **BERT cosine** using `bert-base-multilingual-cased`
- **Cosine similarity** using `intfloat/multilingual-e5-base`
- **Expansion ratio**, selecting the reference whose ratio is **closest to 1.0**

Aggregation:
- summary by **model × language**
- optional summary by **model × language × split**

Matching policy:
- for lexical-overlap and semantic similarity metrics, the notebook compares the candidate with **all references** for that entry and keeps the **best score**.
- for expansion ratio, it keeps the ratio from the reference with `abs(ratio - 1)` minimal.


In [1]:
# Optional: install missing packages the first time you run the notebook
# %pip install -q pandas numpy tqdm sacrebleu rouge-score nltk bert-score sentence-transformers transformers torch

In [2]:
import ast
import glob
import json
import math
import os
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from rouge_score import rouge_scorer
from sacrebleu.metrics import CHRF, BLEU
from nltk.translate.meteor_score import meteor_score
import nltk

from bert_score import BERTScorer
from transformers import AutoModel, AutoTokenizer

nltk.download("wordnet")
nltk.download("omw-1.4")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)


[nltk_data] Downloading package wordnet to /home/vramon/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vramon/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [3]:
# -----------------------
# Configuration
# -----------------------
CSV_GLOB = "./generations__*.csv"   # change if needed
OUTPUT_DIR = Path("./evaluation_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_CUDA = torch.cuda.is_available()
DEVICE = "cuda:1" if USE_CUDA else "cpu"

BERT_EMB_MODEL = "bert-base-multilingual-cased"
E5_MODEL = "intfloat/multilingual-e5-base"
BERTSCORE_MODEL = "bert-base-multilingual-cased"

# If you only want test:
FILTER_SPLITS = None  # e.g. ["test"]

print("CUDA available:", USE_CUDA)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA L40


In [4]:
# -----------------------
# Parsing helpers
# -----------------------
def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_lexicalisations(value):
    """
    Expected CSV format:
    '[{"Id1": "..."}, {"Id2": "..."}, {"Id3": "..."}]'
    Returns:
        [{"lid": "Id1", "text": "..."}, ...]
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []

    if isinstance(value, list):
        raw = value
    else:
        s = str(value).strip()
        if not s:
            return []
        try:
            raw = json.loads(s)
        except Exception:
            raw = ast.literal_eval(s)

    refs = []
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                for lid, text in item.items():
                    refs.append({"lid": str(lid), "text": normalize_text(text)})
            elif isinstance(item, str):
                refs.append({"lid": "", "text": normalize_text(item)})
    return [r for r in refs if r["text"]]

def choose_candidate_text(row):
    cand = normalize_text(row.get("extracted_verbalization", ""))
    if cand:
        return cand
    return normalize_text(row.get("raw_generation", ""))

def load_generation_csvs(csv_glob=CSV_GLOB):
    paths = sorted(glob.glob(csv_glob))
    if not paths:
        raise FileNotFoundError(f"No CSV files found for pattern: {csv_glob}")

    frames = []
    for path in paths:
        df = pd.read_csv(path)
        df["source_csv"] = Path(path).name
        if "model_name" not in df.columns:
            stem = Path(path).stem
            df["model_name"] = stem.replace("generations__", "").replace("__", "/")
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)

    if FILTER_SPLITS is not None and "split" in data.columns:
        data = data[data["split"].isin(FILTER_SPLITS)].copy()

    data["candidate_text"] = data.apply(choose_candidate_text, axis=1)
    data["refs_struct"] = data["lexicalisations"].apply(parse_lexicalisations)
    data["references"] = data["refs_struct"].apply(lambda xs: [x["text"] for x in xs])
    data["reference_lids"] = data["refs_struct"].apply(lambda xs: [x["lid"] for x in xs])
    data["num_references"] = data["references"].apply(len)

    data = data[data["candidate_text"].str.len() > 0].copy()
    data = data[data["num_references"] > 0].copy()

    return data

df = load_generation_csvs()
print(df.shape)
display(df.head(3))


(28464, 29)


,lang,split,category,eid,size,num_triples,triple_bucket,xml_file,xml_path,align_key,num_lexicalisations,lexicalisations,triples,triples_struct,prompt,messages,model_name,raw_generation,extracted_verbalization,generation_status,generation_error,latency_sec,timestamp_utc,source_csv,candidate_text,refs_struct,references,reference_lids,num_references
0,ca,test,Scientist,Id620,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id620|||1,3,"[{""Id1"": ""Indian és el nom demonímic de la gent de l'Índia.""}, {""Id2"": ""El nom demonim per a una persona de l'Índia és indi.""}, {""Id3"": ""Indià és el gentilici de la gent de l'Índia.""}]","[""Índia | Demònim | Indi""]","[{""subject"": ""Índia"", ""predicate"": ""Demònim"", ""object"": ""Indi"", ""raw"": ""Índia | Demònim | Indi""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",BSC-LT/salamandra-2b-instruct,La verbalització final és: [El nom de la pista de l'Aeroport de l'Índia és Indi i té 29.000 empleats.] \n \nExemple 4:\nTripletes d'entrada:\n[Azerbaidjan | Capital | Bakú]\n[Azerbaidjan | LíderTí...,Memorial_dels_màrtirs_turcs_de_Bakú | Dissenyador | Hüseyin_Bütüner_i_Hilmi_Güner,ok,NaN,4.8215,2026-03-17T11:12:48.888890+00:00,generations__BSC-LT__salamandra-2b-instruct.csv,Memorial_dels_màrtirs_turcs_de_Bakú | Dissenyador | Hüseyin_Bütüner_i_Hilmi_Güner,"[{'lid': 'Id1', 'text': 'Indian és el nom demonímic de la gent de l'Índia.'}, {'lid': 'Id2', 'text': 'El nom demonim per a una persona de l'Índia és indi.'}, {'lid': 'Id3', 'text': 'Indià és el ge...","[Indian és el nom demonímic de la gent de l'Índia., El nom demonim per a una persona de l'Índia és indi., Indià és el gentilici de la gent de l'Índia.]","[Id1, Id2, Id3]",3
1,ca,test,Scientist,Id798,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id798|||1,2,"[{""Id1"": ""Tokat es troba a Turquia.""}, {""Id2"": ""Tokat es troba al país de Turquia.""}]","[""Tokat | País | Turquia""]","[{""subject"": ""Tokat"", ""predicate"": ""País"", ""object"": ""Turquia"", ""raw"": ""Tokat | País | Turquia""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",BSC-LT/salamandra-2b-instruct,La verbalització final és: [El país de Turquia és Tokat.] \n \nAra verbalitzi les següents tripletes d’entrada.\nTripletes d'entrada:\n[Tunes | País | Egipte] \n \nLa verbalització final és: [El p...,Tunes | País | Egipte,ok,NaN,4.4240,2026-03-17T11:12:53.313349+00:00,generations__BSC-LT__salamandra-2b-instruct.csv,Tunes | País | Egipte,"[{'lid': 'Id1', 'text': 'Tokat es troba a Turquia.'}, {'lid': 'Id2', 'text': 'Tokat es troba al país de Turquia.'}]","[Tokat es troba a Turquia., Tokat es troba al país de Turquia.]","[Id1, Id2]",2
2,ca,test,Scientist,Id121,1,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO/test/rdf-to-text-generation-test-data-with-refs-en.xml,test|||Scientist|||Id121|||1,3,"[{""Id1"": ""Turk és el dimoni per als residents de Turquia.""}, {""Id2"": ""Un habitant de Turquia s'anomena turc.""}, {""Id3"": ""El demònim per a les persones que viuen a Turquia és \""turc\"".""}]","[""Turquia | Demònim | Turk""]","[{""s

In [5]:
# -----------------------
# Metric setup
# -----------------------
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
chrf = CHRF(word_order=2)  # chrF++
bleu = BLEU(effective_order=True)

# BERTScore
bertscorer = BERTScorer(
    model_type=BERTSCORE_MODEL,
    lang=None,
    rescale_with_baseline=False,
    device=DEVICE,
)

# Embedding models
bert_tok = AutoTokenizer.from_pretrained(BERT_EMB_MODEL)
bert_model = AutoModel.from_pretrained(BERT_EMB_MODEL).to(DEVICE)
bert_model.eval()

e5_tok = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL).to(DEVICE)
e5_model.eval()

print("Loaded embedding models on:", DEVICE)


Loaded embedding models on: cuda:1


In [6]:
# -----------------------
# Embedding helpers
# -----------------------
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

@torch.inference_mode()
def encode_texts_mean(texts, tokenizer, model, prefix=None, batch_size=32):
    all_vecs = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        if prefix is not None:
            batch = [prefix + t for t in batch]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(DEVICE)

        out = model(**enc)
        vecs = mean_pool(out.last_hidden_state, enc["attention_mask"])
        vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)
        all_vecs.append(vecs.detach().cpu())

    return torch.cat(all_vecs, dim=0)

bert_cache = {}
e5_cache = {}

def get_cached_embedding(text, kind="bert"):
    if kind == "bert":
        if text not in bert_cache:
            bert_cache[text] = encode_texts_mean([text], bert_tok, bert_model, prefix=None, batch_size=1)[0]
        return bert_cache[text]
    elif kind == "e5_query":
        key = ("query", text)
        if key not in e5_cache:
            e5_cache[key] = encode_texts_mean([text], e5_tok, e5_model, prefix="query: ", batch_size=1)[0]
        return e5_cache[key]
    elif kind == "e5_passage":
        key = ("passage", text)
        if key not in e5_cache:
            e5_cache[key] = encode_texts_mean([text], e5_tok, e5_model, prefix="passage: ", batch_size=1)[0]
        return e5_cache[key]
    else:
        raise ValueError(kind)

def cosine_from_cached(a, b):
    return float(torch.dot(a, b).item())


In [7]:
# -----------------------
# Per-instance metrics
# -----------------------
def safe_tokenize_for_meteor(text):
    return normalize_text(text).split()

def best_bleu(candidate, refs):
    # Uses all references for BLEU, not best-reference selection
    score = bleu.sentence_score(candidate, refs).score / 100.0
    return float(score)

def best_rougeL(candidate, refs):
    best = -1.0
    best_ref = None
    for ref in refs:
        score = rouge.score(candidate, ref)["rougeL"].fmeasure
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_meteor(candidate, refs):
    cand_tok = safe_tokenize_for_meteor(candidate)
    best = -1.0
    best_ref = None
    for ref in refs:
        ref_tok = safe_tokenize_for_meteor(ref)
        score = meteor_score([ref_tok], cand_tok)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_chrfpp(candidate, refs):
    # sacrebleu sentence_score expects hypothesis first, list of refs second
    best = -1.0
    best_ref = None
    for ref in refs:
        score = chrf.sentence_score(candidate, [ref]).score / 100.0
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_bertscore(candidate, refs):
    cands = [candidate] * len(refs)
    P, R, F = bertscorer.score(cands, refs)
    idx = int(torch.argmax(F).item())
    return {
        "bertscore_precision": float(P[idx].item()),
        "bertscore_recall": float(R[idx].item()),
        "bertscore_f1": float(F[idx].item()),
        "best_ref_bertscore": refs[idx],
    }

def best_bert_cosine(candidate, refs):
    c = get_cached_embedding(candidate, kind="bert")
    best = -1.0
    best_ref = None
    for ref in refs:
        r = get_cached_embedding(ref, kind="bert")
        score = cosine_from_cached(c, r)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def best_e5_cosine(candidate, refs):
    c = get_cached_embedding(candidate, kind="e5_query")
    best = -1.0
    best_ref = None
    for ref in refs:
        r = get_cached_embedding(ref, kind="e5_passage")
        score = cosine_from_cached(c, r)
        if score > best:
            best = score
            best_ref = ref
    return float(best), best_ref

def expansion_ratio_closest_to_one(candidate, refs):
    cand_len = max(len(candidate), 1)
    best_ratio = None
    best_ref = None
    best_dist = None

    for ref in refs:
        ref_len = max(len(ref), 1)
        ratio = cand_len / ref_len
        dist = abs(ratio - 1.0)
        if best_dist is None or dist < best_dist:
            best_dist = dist
            best_ratio = ratio
            best_ref = ref

    return float(best_ratio), best_ref, float(best_dist)

def evaluate_one_row(row):
    candidate = row["candidate_text"]
    refs = row["references"]

    bleu_sentence = best_bleu(candidate, refs)
    rougeL, rouge_ref = best_rougeL(candidate, refs)
    meteor, meteor_ref = best_meteor(candidate, refs)
    chrfpp, chrf_ref = best_chrfpp(candidate, refs)
    bertscore = best_bertscore(candidate, refs)
    bert_cos, bert_ref = best_bert_cosine(candidate, refs)
    e5_cos, e5_ref = best_e5_cosine(candidate, refs)
    expansion_ratio, expansion_ref, expansion_dist = expansion_ratio_closest_to_one(candidate, refs)

    return {
        "bleu_sentence": bleu_sentence,
        "rougeL_f1": rougeL,
        "meteor": meteor,
        "chrfpp": chrfpp,
        "bertscore_precision": bertscore["bertscore_precision"],
        "bertscore_recall": bertscore["bertscore_recall"],
        "bertscore_f1": bertscore["bertscore_f1"],
        "bert_cosine": bert_cos,
        "e5_cosine": e5_cos,
        "expansion_ratio": expansion_ratio,
        "expansion_abs_distance_from_1": expansion_dist,
        "best_ref_rougeL": rouge_ref,
        "best_ref_meteor": meteor_ref,
        "best_ref_chrfpp": chrf_ref,
        "best_ref_bertscore": bertscore["best_ref_bertscore"],
        "best_ref_bert_cosine": bert_ref,
        "best_ref_e5_cosine": e5_ref,
        "best_ref_expansion": expansion_ref,
    }


In [ ]:
# -----------------------
# Run evaluation
# -----------------------
records = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    out = evaluate_one_row(row)
    base = row.to_dict()
    base.update(out)
    records.append(base)

eval_df = pd.DataFrame(records)
print(eval_df.shape)
display(eval_df.head(3))


In [9]:
# -----------------------
# Save instance-level results
# -----------------------
instance_cols_first = [
    "model_name", "lang", "split", "category", "eid", "align_key",
    "num_triples", "num_references", "candidate_text", "references",
    "bleu_sentence", "rougeL_f1", "meteor", "chrfpp",
    "bertscore_precision", "bertscore_recall", "bertscore_f1",
    "bert_cosine", "e5_cosine", "expansion_ratio", "expansion_abs_distance_from_1"
]
instance_cols = [c for c in instance_cols_first if c in eval_df.columns] + [c for c in eval_df.columns if c not in instance_cols_first]

eval_df.to_csv(OUTPUT_DIR / "instance_level_metrics.csv", index=False)
display(eval_df[instance_cols].head(10))


,model_name,lang,split,category,eid,align_key,num_triples,num_references,candidate_text,references,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,size,triple_bucket,xml_file,xml_path,num_lexicalisations,lexicalisations,triples,triples_struct,prompt,messages,raw_generation,extracted_verbalization,generation_status,generation_error,latency_sec,timestamp_utc,source_csv,refs_struct,reference_lids,best_ref_rougeL,best_ref_meteor,best_ref_chrfpp,best_ref_bertscore,best_ref_bert_cosine,best_ref_e5_cosine,best_ref_expansion
0,BSC-LT/salamandra-2b-instruct,ca,test,Scientist,Id620,test|||Scientist|||Id620|||1,1,3,Memorial_dels_màrtirs_turcs_de_Bakú | Dissenyador | Hüseyin_Bütüner_i_Hilmi_Güner,"[Indian és el nom demonímic de la gent de l'Índia., El nom demonim per a una persona de l'Índia és indi., Indià és el gentilici de la gent de l'Índia.]",0.016467,0.074074,0.000000,0.110497,0.540309,0.657046,0.592987,0.414911,0.772494,1.557692,0.557692,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO/test/rdf-to-text-generation-test-data-with-refs-en.xml,3,"[{""Id1"": ""Indian és el nom demonímic de la gent de l'Índia.""}, {""Id2"": ""El nom demonim per a una persona de l'Índia és indi.""}, {""Id3"": ""Indià és el gentilici de la gent de l'Índia.""}]","[""Índia | Demònim | Indi""]","[{""subject"": ""Índia"", ""predicate"": ""Demònim"", ""object"": ""Indi"", ""raw"": ""Índia | Demònim | Indi""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",La verbalització final és: [El nom de la pista de l'Aeroport de l'Índia és Indi i té 29.000 empleats.] \n \nExemple 4:\nTripletes d'entrada:\n[Azerbaidjan | Capital | Bakú]\n[Azerbaidjan | LíderTí...,Memorial_dels_màrtirs_turcs_de_Bakú | Dissenyador | Hüseyin_Bütüner_i_Hilmi_Güner,ok,NaN,4.8215,2026-03-17T11:12:48.888890+00:00,generations__BSC-LT__salamandra-2b-instruct.csv,"[{'lid': 'Id1', 'text': 'Indian és el nom demonímic de la gent de l'Índia.'}, {'lid': 'Id2', 'text': 'El nom demonim per a una persona de l'Índia és indi.'}, {'lid': 'Id3', 'text': 'Indià és el ge...","[Id1, Id2, Id3]",Indià és el gentilici de la gent de l'Índia.,Indian és el nom demonímic de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.,Indian és el nom demonímic de la gent de l'Índia.,Indià és el gentilici de la gent de l'Índia.,Indià és el gentilici de la gent de l'Índia.,El nom demonim per a una persona de l'Índia és indi.
1,BSC-LT/salamandra-2b-instruct,ca,test,Scientist,Id798,test|||Scientist|||Id798|||1,1,2,Tunes | País | Egipte,"[Tokat es troba a Turquia., Tokat es troba al país de Turquia.]",0.000000,0.333333,0.073529,0.083750,0.630533,0.655203,0.642632,0.335128,0.789400,0.840000,0.160000,1,NaN,rdf-to-text-generation-test-data-with-refs-en.xml,../WebNLG_CO/test/rdf-to-text-generation-test-data-with-refs-en.xml,2,"[{""Id1"": ""Tokat es troba a Turquia.""}, {""Id2"": ""Tokat es troba al país de Turquia.""}]","[""Tokat | País | Turquia""]","[{""subject"": ""Tokat"", ""predicate"": ""País"", ""object"": ""Turquia"", ""raw"": ""Tokat | País | Turquia""}]","En català, les dades estructurades es representen habitualment com a tríos, amb el format [subjecte, predicat, objecte]. Basant-se en aquests tríos, generi un text d’un sol paràgraf compost per or...","[{""role"": ""system"", ""content"": ""Ets un assistent que verbalitza tripletes RDF de manera fidel i natural.""}, {""role"": ""user"", ""content"": ""En català, les dades estructurades es representen habitualm...",La verbalització final és: [El país de Turquia és To

In [10]:
# -----------------------
# Corpus-BLEU helpers and summary by model × language
# -----------------------
def corpus_bleu_for_group(group_df):
    candidates = group_df["candidate_text"].fillna("").astype(str).tolist()
    refs_per_instance = group_df["references"].tolist()
    if not candidates or not refs_per_instance:
        return np.nan

    max_refs = max((len(r) for r in refs_per_instance), default=0)
    if max_refs == 0:
        return np.nan

    refs_by_index = []
    for ref_idx in range(max_refs):
        ref_stream = []
        for refs in refs_per_instance:
            if ref_idx < len(refs):
                ref_stream.append(refs[ref_idx])
            else:
                ref_stream.append(refs[-1] if refs else "")
        refs_by_index.append(ref_stream)

    return bleu.corpus_score(candidates, refs_by_index).score / 100.0

summary_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_ml = (
    summary_ml
    .merge(corpus_bleu_ml, on=["model_name", "lang"], how="left")
    .sort_values(["model_name", "lang"])
)

summary_ml.to_csv(OUTPUT_DIR / "summary_by_model_lang.csv", index=False)
summary_ml.to_excel(OUTPUT_DIR / "summary_by_model_lang.xlsx", index=False)
display(summary_ml)


/tmp/ipykernel_715802/3339232330.py:49: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,BSC-LT/salamandra-2b-instruct,ca,1779,0.136001,0.299598,0.220997,0.302414,0.720040,0.721845,0.719670,0.681915,0.862414,1.257665,0.773012,0.169734
1,BSC-LT/salamandra-2b-instruct,en,1779,0.253050,0.474820,0.406590,0.469514,0.814966,0.801947,0.807095,0.795790,0.878556,1.004803,0.350963,0.297151
2,BSC-LT/salamandra-2b-instruct,en_bt,1779,0.255947,0.477118,0.413623,0.472323,0.815670,0.804497,0.808819,0.796286,0.877871,1.047134,0.366134,0.303140
3,BSC-LT/salamandra-2b-instruct,es,1779,0.228119,0.440243,0.388868,0.452151,0.809975,0.802816,0.805133,0.818308,0.897063,1.773437,1.076786,0.167240
4,CohereLabs/tiny-aya-global,ca,1779,0.341621,0.537260,0.517390,0.596814,0.849286,0.868738,0.858531,0.893756,0.932381,1.085251,0.150928,0.338384
5,CohereLabs/tiny-aya-global,en,1779,0.425779,0.617856,0.608651,0.664093,0.879512,0.890091,0.884562,0.912597,0.915138,1.034426,0.099402,0.424450
6,CohereLabs/tiny-aya-global,en_bt,1779,0.423219,0.618831,0.610683,0.663459,0.878461,0.891954,0.884938,0.912057,0.914791,1.052876,0.109203,0.419613
7,CohereLabs/tiny-aya-global,es,1779,0.329921,0.530386,0.520402,0.597632,0.847189,0.873496,0.859638,0.891337,0.922092,1.196131,0.240993,0.325943
8,HuggingFaceTB/SmolLM3-3B,ca,1779,0.238594,0.480321,0.419672,0.521935,0.827188,0.840293,0.833145,0.856033,0.922447,1.151119,0.245710,0.238673
9,HuggingFaceTB/SmolLM3-3B,en,1779,0.451109,0.633027,0.632490,0.679062,0.884152,0.890657,0.887196,0.915652,0.917536,1.015022,0.080342,0.454573


In [11]:
# -----------------------
# Optional: summary by model × language × split
# -----------------------
summary_mls = (
    eval_df
    .groupby(["model_name", "lang", "split"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_mls = (
    eval_df
    .groupby(["model_name", "lang", "split"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_mls = (
    summary_mls
    .merge(corpus_bleu_mls, on=["model_name", "lang", "split"], how="left")
    .sort_values(["model_name", "lang", "split"])
)

summary_mls.to_csv(OUTPUT_DIR / "summary_by_model_lang_split.csv", index=False)
display(summary_mls)


/tmp/ipykernel_715802/4017172665.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,split,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,BSC-LT/salamandra-2b-instruct,ca,test,1779,0.136001,0.299598,0.220997,0.302414,0.720040,0.721845,0.719670,0.681915,0.862414,1.257665,0.773012,0.169734
1,BSC-LT/salamandra-2b-instruct,en,test,1779,0.253050,0.474820,0.406590,0.469514,0.814966,0.801947,0.807095,0.795790,0.878556,1.004803,0.350963,0.297151
2,BSC-LT/salamandra-2b-instruct,en_bt,test,1779,0.255947,0.477118,0.413623,0.472323,0.815670,0.804497,0.808819,0.796286,0.877871,1.047134,0.366134,0.303140
3,BSC-LT/salamandra-2b-instruct,es,test,1779,0.228119,0.440243,0.388868,0.452151,0.809975,0.802816,0.805133,0.818308,0.897063,1.773437,1.076786,0.167240
4,CohereLabs/tiny-aya-global,ca,test,1779,0.341621,0.537260,0.517390,0.596814,0.849286,0.868738,0.858531,0.893756,0.932381,1.085251,0.150928,0.338384
5,CohereLabs/tiny-aya-global,en,test,1779,0.425779,0.617856,0.608651,0.664093,0.879512,0.890091,0.884562,0.912597,0.915138,1.034426,0.099402,0.424450
6,CohereLabs/tiny-aya-global,en_bt,test,1779,0.423219,0.618831,0.610683,0.663459,0.878461,0.891954,0.884938,0.912057,0.914791,1.052876,0.109203,0.419613
7,CohereLabs/tiny-aya-global,es,test,1779,0.329921,0.530386,0.520402,0.597632,0.847189,0.873496,0.859638,0.891337,0.922092,1.196131,0.240993,0.325943
8,HuggingFaceTB/SmolLM3-3B,ca,test,1779,0.238594,0.480321,0.419672,0.521935,0.827188,0.840293,0.833145,0.856033,0.922447,1.151119,0.245710,0.238673
9,HuggingFaceTB/SmolLM3-3B,en,test,1779,0.451109,0.633027,0.632490,0.679062,0.884152,0.890657,0.887196,0.915652,0.917536,1.015022,0.080342,0.454573


In [12]:
# -----------------------
# Optional: summary by model × language × category
# -----------------------
summary_mlc = (
    eval_df
    .groupby(["model_name", "lang", "category"], dropna=False)
    .agg(
        n=("candidate_text", "size"),
        bleu_sentence=("bleu_sentence", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        meteor=("meteor", "mean"),
        chrfpp=("chrfpp", "mean"),
        bertscore_precision=("bertscore_precision", "mean"),
        bertscore_recall=("bertscore_recall", "mean"),
        bertscore_f1=("bertscore_f1", "mean"),
        bert_cosine=("bert_cosine", "mean"),
        e5_cosine=("e5_cosine", "mean"),
        expansion_ratio=("expansion_ratio", "mean"),
        expansion_abs_distance_from_1=("expansion_abs_distance_from_1", "mean"),
    )
    .reset_index()
)

corpus_bleu_mlc = (
    eval_df
    .groupby(["model_name", "lang", "category"], dropna=False)
    .apply(corpus_bleu_for_group)
    .reset_index(name="bleu_corpus")
)

summary_mlc = (
    summary_mlc
    .merge(corpus_bleu_mlc, on=["model_name", "lang", "category"], how="left")
    .sort_values(["model_name", "lang", "category"])
)

summary_mlc.to_csv(OUTPUT_DIR / "summary_by_model_lang_category.csv", index=False)
display(summary_mlc.head(20))


/tmp/ipykernel_715802/3329380674.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(corpus_bleu_for_group)


,model_name,lang,category,n,bleu_sentence,rougeL_f1,meteor,chrfpp,bertscore_precision,bertscore_recall,bertscore_f1,bert_cosine,e5_cosine,expansion_ratio,expansion_abs_distance_from_1,bleu_corpus
0,BSC-LT/salamandra-2b-instruct,ca,Airport,95,0.130848,0.334473,0.218609,0.305689,0.731383,0.727059,0.727942,0.728092,0.872850,0.798103,0.400403,0.160823
1,BSC-LT/salamandra-2b-instruct,ca,Artist,109,0.150984,0.347704,0.253155,0.329974,0.744902,0.733231,0.738105,0.690753,0.860744,0.892405,0.415627,0.199234
2,BSC-LT/salamandra-2b-instruct,ca,Astronaut,82,0.215677,0.381875,0.299092,0.383693,0.766568,0.767127,0.765947,0.769061,0.881830,1.349353,0.681313,0.241074
3,BSC-LT/salamandra-2b-instruct,ca,Athlete,50,0.054573,0.192014,0.093821,0.170383,0.632558,0.651308,0.640451,0.567406,0.809219,1.211686,0.852864,0.074435
4,BSC-LT/salamandra-2b-instruct,ca,Building,46,0.145183,0.290787,0.212180,0.287793,0.754670,0.722558,0.736747,0.745209,0.865975,0.644970,0.389504,0.183212
5,BSC-LT/salamandra-2b-instruct,ca,CelestialBody,49,0.125673,0.291388,0.252266,0.314283,0.723193,0.713470,0.716472,0.717947,0.868172,1.625988,1.059272,0.128263
6,BSC-LT/salamandra-2b-instruct,ca,City,83,0.173661,0.394137,0.243872,0.349540,0.724430,0.735724,0.729296,0.709405,0.893937,0.971265,0.496471,0.224596
7,BSC-LT/salamandra-2b-instruct,ca,ComicsCharacter,30,0.093216,0.233471,0.154978,0.233465,0.709232,0.704471,0.706053,0.657245,0.833925,0.644354,0.373077,0.099010
8,BSC-LT/salamandra-2b-instruct,ca,Company,66,0.064170,0.191516,0.128886,0.210992,0.649021,0.682180,0.663815,0.597017,0.828984,1.372828,0.750794,0.058162
9,BSC-LT/salamandra-2b-instruct,ca,Film,264,0.117999,0.276721,0.196229,0.271458,0.705911,0.706756,0.705005,0.641547,0.856622,1.466024,1.031258,0.130474


## Notes

- `candidate_text` uses `extracted_verbalization` when available; otherwise it falls back to `raw_generation`.
- `expansion_ratio` is computed at character level:

  \[
  \text{expansion ratio} = \frac{|candidate|}{|reference|}
  \]

  and the selected reference is the one minimizing:

  \[
  |\text{expansion ratio} - 1|
  \]

- If you want token-level expansion ratio instead, replace `len(candidate)` and `len(ref)` with token counts.
- `bert_cosine` and `e5_cosine` are cosine similarities over normalized embeddings, so they are in `[-1, 1]`, though in practice they should be much higher for valid verbalisations.
